<a href="https://colab.research.google.com/github/rodrigologin0-cpu/Rodrigo-de-Souza-Lima/blob/main/NSGA_II_%2B_BUSCA_ALEAT%C3%93RIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NSGA-II + BUSCA ALEATÓRIA PARA SELEÇÃO ESTRUTURAL NARX
# ============================================================
# Objetivo:
#   1) Montar um banco de regressores NARX a partir de entradas u(k) e saída y(k).
#   2) Usar NSGA-II para selecionar subconjuntos de termos.
#   3) Otimizar os hiperparâmetros do NSGA-II por busca aleatória externa.
#   4) Retornar uma fronteira de Pareto RMSE x Complexidade.
#
# Autor: estrutura didática para aplicação industrial / acadêmica.
# Dependências: numpy, pandas, matplotlib, scikit-learn opcional não usado.
# ============================================================

import os
import time
import math
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURAÇÕES PRINCIPAIS
# ============================================================

CONFIG = {
    # --------------------------------------------------------
    # Arquivo de dados
    # --------------------------------------------------------
    "ARQUIVO_DADOS": "dados_narx.xlsx",

    # Se o arquivo acima não existir, gera dados sintéticos para testar o código.
    "USAR_DADOS_EXEMPLO_SE_NAO_EXISTIR": True,

    # --------------------------------------------------------
    # Colunas do problema
    # Ajuste estes nomes para o seu Excel.
    # --------------------------------------------------------
    "COLUNA_SAIDA": "y",
    "COLUNAS_ENTRADA": ["u1", "u2"],

    # --------------------------------------------------------
    # Configuração NARX
    # --------------------------------------------------------
    "MAX_LAG_Y": 4,
    "MAX_LAG_U": 4,

    # Grau dos termos candidatos:
    # 1 = apenas termos lineares atrasados.
    # 2 = inclui quadráticos e interações entre regressores.
    "GRAU_POLINOMIAL": 2,

    # Limite de termos polinomiais para evitar explosão combinatória.
    # Se o número de termos candidatos for muito grande, o algoritmo fica pesado.
    "MAX_FEATURES_POLINOMIAIS": 250,

    # Penalização Ridge para estimar os coeficientes do modelo.
    # Ajuda a evitar instabilidade numérica quando há muitos regressores correlacionados.
    "RIDGE_ALPHA": 1e-5,

    # Divisão temporal treino/validação.
    # Para série temporal, não embaralhar.
    "FRAC_TREINO": 0.70,

    # Número máximo de termos que um indivíduo pode selecionar.
    # Ajuda a manter modelos viáveis para aplicação prática.
    "MAX_TERMOS_MODELO": 35,

    # --------------------------------------------------------
    # Busca aleatória dos hiperparâmetros do NSGA-II
    # --------------------------------------------------------
    "N_TENTATIVAS_BUSCA_ALEATORIA": 10,

    # Para reduzir tempo, durante a busca usa-se menos gerações.
    # Depois o melhor conjunto de hiperparâmetros é rodado novamente com mais gerações.
    "MULTIPLICADOR_GERACOES_FINAL": 2,

    # Semente para reprodutibilidade.
    "SEED_GLOBAL": 42,

    # --------------------------------------------------------
    # Saídas
    # --------------------------------------------------------
    "PASTA_SAIDA": "resultados_nsga2_narx",
}


# ============================================================
# 2. ESTRUTURAS DE DADOS
# ============================================================

@dataclass
class NSGAParams:
    pop_size: int
    n_gen: int
    crossover_prob: float
    mutation_prob_gene: float
    init_prob_gene: float
    tournament_size: int
    seed: int


@dataclass
class Individual:
    mask: np.ndarray
    objectives: Optional[np.ndarray] = None
    rank: Optional[int] = None
    crowding: float = 0.0


# ============================================================
# 3. UTILITÁRIOS GERAIS
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def garantir_pasta(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def normalizar_treino_validacao(X_train: np.ndarray, X_val: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    return (X_train - mu) / sigma, (X_val - mu) / sigma, mu, sigma


def resolver_colunas(df: pd.DataFrame, coluna_saida: str, colunas_entrada: List[str]) -> None:
    faltantes = []
    if coluna_saida not in df.columns:
        faltantes.append(coluna_saida)
    for c in colunas_entrada:
        if c not in df.columns:
            faltantes.append(c)

    if faltantes:
        raise ValueError(
            "Colunas não encontradas no arquivo: " + str(faltantes) +
            "\nColunas disponíveis: " + str(list(df.columns))
        )


# ============================================================
# 4. DADOS DE EXEMPLO
# ============================================================

def gerar_dados_exemplo(n: int = 2500, seed: int = 42) -> pd.DataFrame:
    """
    Gera uma série sintética com comportamento não linear tipo NARX.
    Serve apenas para testar o script quando o Excel ainda não foi configurado.
    """
    set_seed(seed)

    u1 = np.random.normal(0, 1, n)
    u2 = np.random.normal(0, 1, n)
    y = np.zeros(n)

    ruido = np.random.normal(0, 0.05, n)

    for k in range(4, n):
        y[k] = (
            0.62 * y[k - 1]
            - 0.18 * y[k - 2]
            + 0.35 * u1[k - 1]
            + 0.15 * u2[k - 2]
            + 0.10 * y[k - 1] * u1[k - 2]
            - 0.08 * (u1[k - 3] ** 2)
            + ruido[k]
        )

    return pd.DataFrame({"u1": u1, "u2": u2, "y": y})


# ============================================================
# 5. CONSTRUÇÃO DOS REGRESSORES NARX
# ============================================================

def criar_matriz_lags(
    df: pd.DataFrame,
    coluna_saida: str,
    colunas_entrada: List[str],
    max_lag_y: int,
    max_lag_u: int,
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Cria regressores atrasados:
        y(k-1), y(k-2), ...
        u1(k-1), u1(k-2), ...
        u2(k-1), u2(k-2), ...
    O alvo é y(k).
    """
    dados = pd.DataFrame(index=df.index)

    for lag in range(1, max_lag_y + 1):
        dados[f"{coluna_saida}(k-{lag})"] = df[coluna_saida].shift(lag)

    for col in colunas_entrada:
        for lag in range(1, max_lag_u + 1):
            dados[f"{col}(k-{lag})"] = df[col].shift(lag)

    alvo = df[coluna_saida].copy()

    conjunto = pd.concat([dados, alvo.rename("target")], axis=1).dropna()
    X_base = conjunto.drop(columns=["target"])
    y = conjunto["target"]

    return X_base, y


def criar_features_polinomiais(
    X_base: pd.DataFrame,
    grau: int = 1,
    max_features: int = 250,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Cria termos polinomiais candidatos.
    Grau 1: mantém apenas regressores lineares.
    Grau 2: adiciona quadrados e interações entre pares.

    Observação:
    Para muitos regressores, os termos quadráticos crescem rapidamente.
    Por isso há um limite max_features.
    """
    if grau <= 1:
        return X_base.copy()

    set_seed(seed)
    X = X_base.copy()
    cols = list(X_base.columns)

    novas_features = {}

    # Quadrados
    for c in cols:
        nome = f"{c}^2"
        novas_features[nome] = X_base[c].values * X_base[c].values

    # Interações
    pares = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pares.append((cols[i], cols[j]))

    random.shuffle(pares)

    for c1, c2 in pares:
        nome = f"{c1}*{c2}"
        novas_features[nome] = X_base[c1].values * X_base[c2].values

    X_poly = pd.concat([X, pd.DataFrame(novas_features, index=X_base.index)], axis=1)

    if X_poly.shape[1] > max_features:
        # Mantém todos os lineares e seleciona aleatoriamente parte dos não lineares.
        cols_lineares = cols
        cols_extra = [c for c in X_poly.columns if c not in cols_lineares]
        n_extra_permitido = max_features - len(cols_lineares)

        if n_extra_permitido < 0:
            raise ValueError("max_features menor que o número de termos lineares.")

        cols_extra_escolhidas = random.sample(cols_extra, n_extra_permitido)
        X_poly = X_poly[cols_lineares + cols_extra_escolhidas]

    return X_poly


# ============================================================
# 6. ESTIMAÇÃO DO MODELO NARX POR RIDGE
# ============================================================

def ridge_fit_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    alpha: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Ajusta regressão Ridge por forma fechada:
        beta = (X'X + alpha I)^-1 X'y
    Inclui intercepto não penalizado.
    """
    Xtr = np.column_stack([np.ones(X_train.shape[0]), X_train])
    Xva = np.column_stack([np.ones(X_val.shape[0]), X_val])

    n_coef = Xtr.shape[1]
    I = np.eye(n_coef)
    I[0, 0] = 0.0  # não penalizar intercepto

    A = Xtr.T @ Xtr + alpha * I
    b = Xtr.T @ y_train

    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b

    yhat_train = Xtr @ beta
    yhat_val = Xva @ beta

    return yhat_train, yhat_val


# ============================================================
# 7. AVALIAÇÃO DE INDIVÍDUOS
# ============================================================

class Evaluator:
    def __init__(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        feature_names: List[str],
        ridge_alpha: float,
        max_terms: int,
    ):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.feature_names = feature_names
        self.ridge_alpha = ridge_alpha
        self.max_terms = max_terms
        self.cache: Dict[bytes, np.ndarray] = {}
        self.cache_extra: Dict[bytes, Dict] = {}

        # Referência para normalização: predição ingênua pela média do treino.
        y_media = np.full_like(y_val, np.mean(y_train), dtype=float)
        self.rmse_ref = max(rmse(y_val, y_media), 1e-9)

    def reparar_mask(self, mask: np.ndarray) -> np.ndarray:
        mask = mask.astype(bool).copy()

        # Garante pelo menos um termo.
        if mask.sum() == 0:
            idx = np.random.randint(0, len(mask))
            mask[idx] = True

        # Limita quantidade máxima de termos.
        if mask.sum() > self.max_terms:
            ativos = np.where(mask)[0]
            remover = np.random.choice(ativos, size=int(mask.sum() - self.max_terms), replace=False)
            mask[remover] = False

        return mask

    def evaluate(self, mask: np.ndarray) -> np.ndarray:
        mask = self.reparar_mask(mask)
        key = mask.tobytes()

        if key in self.cache:
            return self.cache[key]

        idx = np.where(mask)[0]
        n_terms = len(idx)

        if n_terms == 0:
            obj = np.array([1e9, 1e9], dtype=float)
            self.cache[key] = obj
            return obj

        Xtr_sel = self.X_train[:, idx]
        Xva_sel = self.X_val[:, idx]

        try:
            yhat_train, yhat_val = ridge_fit_predict(
                Xtr_sel, self.y_train, Xva_sel, alpha=self.ridge_alpha
            )

            erro_val = rmse(self.y_val, yhat_val)
            erro_train = rmse(self.y_train, yhat_train)
            gap = abs(erro_val - erro_train)

            # Objetivos minimizados:
            # f1 = RMSE de validação
            # f2 = quantidade de termos
            # Observação: gap é salvo como informação extra, mas não usado como objetivo principal.
            obj = np.array([erro_val, float(n_terms)], dtype=float)

            self.cache_extra[key] = {
                "rmse_train": erro_train,
                "rmse_val": erro_val,
                "mae_val": mae(self.y_val, yhat_val),
                "gap_train_val": gap,
                "n_terms": n_terms,
                "selected_features": [self.feature_names[i] for i in idx],
                "yhat_train": yhat_train,
                "yhat_val": yhat_val,
            }

        except Exception:
            obj = np.array([1e9, 1e9], dtype=float)

        self.cache[key] = obj
        return obj

    def detalhes(self, mask: np.ndarray) -> Dict:
        mask = self.reparar_mask(mask)
        key = mask.tobytes()
        if key not in self.cache_extra:
            _ = self.evaluate(mask)
        return self.cache_extra.get(key, {})


# ============================================================
# 8. FUNÇÕES DO NSGA-II
# ============================================================

def dominates(a: np.ndarray, b: np.ndarray) -> bool:
    """Minimização: a domina b se a <= b em tudo e a < b em pelo menos um objetivo."""
    return bool(np.all(a <= b) and np.any(a < b))


def non_dominated_sort(population: List[Individual]) -> List[List[int]]:
    n = len(population)
    S = [[] for _ in range(n)]
    n_dom = np.zeros(n, dtype=int)
    fronts = [[]]

    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            if dominates(population[p].objectives, population[q].objectives):
                S[p].append(q)
            elif dominates(population[q].objectives, population[p].objectives):
                n_dom[p] += 1

        if n_dom[p] == 0:
            population[p].rank = 0
            fronts[0].append(p)

    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in S[p]:
                n_dom[q] -= 1
                if n_dom[q] == 0:
                    population[q].rank = i + 1
                    next_front.append(q)
        i += 1
        fronts.append(next_front)

    fronts.pop()
    return fronts


def assign_crowding_distance(population: List[Individual], front: List[int]) -> None:
    if len(front) == 0:
        return

    n_obj = len(population[front[0]].objectives)

    for idx in front:
        population[idx].crowding = 0.0

    if len(front) <= 2:
        for idx in front:
            population[idx].crowding = float("inf")
        return

    for m in range(n_obj):
        front_sorted = sorted(front, key=lambda idx: population[idx].objectives[m])

        population[front_sorted[0]].crowding = float("inf")
        population[front_sorted[-1]].crowding = float("inf")

        f_min = population[front_sorted[0]].objectives[m]
        f_max = population[front_sorted[-1]].objectives[m]

        if abs(f_max - f_min) < 1e-12:
            continue

        for i in range(1, len(front_sorted) - 1):
            prev_f = population[front_sorted[i - 1]].objectives[m]
            next_f = population[front_sorted[i + 1]].objectives[m]
            population[front_sorted[i]].crowding += (next_f - prev_f) / (f_max - f_min)


def avaliar_populacao(population: List[Individual], evaluator: Evaluator) -> None:
    for ind in population:
        ind.mask = evaluator.reparar_mask(ind.mask)
        ind.objectives = evaluator.evaluate(ind.mask)


def preparar_rank_crowding(population: List[Individual]) -> None:
    fronts = non_dominated_sort(population)
    for front in fronts:
        assign_crowding_distance(population, front)


def comparar_individuos(a: Individual, b: Individual) -> Individual:
    if a.rank < b.rank:
        return a
    if b.rank < a.rank:
        return b
    if a.crowding > b.crowding:
        return a
    if b.crowding > a.crowding:
        return b
    return a if random.random() < 0.5 else b


def tournament_selection(population: List[Individual], tournament_size: int) -> Individual:
    candidatos = random.sample(population, tournament_size)
    melhor = candidatos[0]
    for c in candidatos[1:]:
        melhor = comparar_individuos(melhor, c)
    return melhor


def uniform_crossover(mask1: np.ndarray, mask2: np.ndarray, crossover_prob: float) -> Tuple[np.ndarray, np.ndarray]:
    if random.random() > crossover_prob:
        return mask1.copy(), mask2.copy()

    troca = np.random.rand(len(mask1)) < 0.5
    child1 = mask1.copy()
    child2 = mask2.copy()

    child1[troca] = mask2[troca]
    child2[troca] = mask1[troca]

    return child1, child2


def bitflip_mutation(mask: np.ndarray, mutation_prob_gene: float) -> np.ndarray:
    mut = np.random.rand(len(mask)) < mutation_prob_gene
    child = mask.copy()
    child[mut] = ~child[mut]
    return child


def inicializar_populacao(n_features: int, pop_size: int, init_prob_gene: float, evaluator: Evaluator) -> List[Individual]:
    population = []
    for _ in range(pop_size):
        mask = np.random.rand(n_features) < init_prob_gene
        mask = evaluator.reparar_mask(mask)
        population.append(Individual(mask=mask))
    return population


def selecionar_proxima_geracao(combined: List[Individual], pop_size: int) -> List[Individual]:
    fronts = non_dominated_sort(combined)
    next_pop = []

    for front in fronts:
        assign_crowding_distance(combined, front)

        if len(next_pop) + len(front) <= pop_size:
            next_pop.extend([combined[i] for i in front])
        else:
            ordenados = sorted(front, key=lambda idx: combined[idx].crowding, reverse=True)
            falta = pop_size - len(next_pop)
            next_pop.extend([combined[i] for i in ordenados[:falta]])
            break

    return next_pop


def executar_nsga2(
    evaluator: Evaluator,
    n_features: int,
    params: NSGAParams,
    verbose: bool = False,
) -> Tuple[List[Individual], Dict]:
    set_seed(params.seed)
    t0 = time.time()

    population = inicializar_populacao(
        n_features=n_features,
        pop_size=params.pop_size,
        init_prob_gene=params.init_prob_gene,
        evaluator=evaluator,
    )

    avaliar_populacao(population, evaluator)
    preparar_rank_crowding(population)

    historico = []

    for gen in range(params.n_gen):
        offspring = []

        while len(offspring) < params.pop_size:
            p1 = tournament_selection(population, params.tournament_size)
            p2 = tournament_selection(population, params.tournament_size)

            c1_mask, c2_mask = uniform_crossover(p1.mask, p2.mask, params.crossover_prob)
            c1_mask = bitflip_mutation(c1_mask, params.mutation_prob_gene)
            c2_mask = bitflip_mutation(c2_mask, params.mutation_prob_gene)

            c1_mask = evaluator.reparar_mask(c1_mask)
            c2_mask = evaluator.reparar_mask(c2_mask)

            offspring.append(Individual(mask=c1_mask))
            if len(offspring) < params.pop_size:
                offspring.append(Individual(mask=c2_mask))

        avaliar_populacao(offspring, evaluator)

        combined = population + offspring
        avaliar_populacao(combined, evaluator)
        population = selecionar_proxima_geracao(combined, params.pop_size)
        preparar_rank_crowding(population)

        fronts = non_dominated_sort(population)
        f0 = fronts[0]
        best_rmse = min(population[i].objectives[0] for i in f0)
        min_terms = min(population[i].objectives[1] for i in f0)
        n_nd = len(f0)

        historico.append({
            "gen": gen + 1,
            "best_rmse_front0": best_rmse,
            "min_terms_front0": min_terms,
            "n_nd_front0": n_nd,
        })

        if verbose and ((gen + 1) % max(1, params.n_gen // 5) == 0 or gen == 0):
            print(
                f"Geração {gen + 1:4d}/{params.n_gen} | "
                f"melhor RMSE={best_rmse:.6f} | "
                f"menor nº termos={min_terms:.0f} | "
                f"não dominados={n_nd}"
            )

    tempo = time.time() - t0

    info = {
        "tempo_s": tempo,
        "historico": pd.DataFrame(historico),
    }

    return population, info


# ============================================================
# 9. MÉTRICAS DA FRONTEIRA E BUSCA ALEATÓRIA
# ============================================================

def obter_frente_nao_dominada(population: List[Individual]) -> List[Individual]:
    fronts = non_dominated_sort(population)
    if not fronts:
        return []
    return [population[i] for i in fronts[0]]


def filtrar_pontos_unicos(front: List[Individual]) -> np.ndarray:
    pontos = np.array([ind.objectives for ind in front], dtype=float)
    if len(pontos) == 0:
        return pontos
    pontos = np.unique(pontos, axis=0)
    return pontos


def hypervolume_2d_min(pontos: np.ndarray, ref: Tuple[float, float]) -> float:
    """
    Hypervolume 2D para minimização.
    Os pontos devem estar em escala normalizada.
    Ref deve ser pior que os pontos, por exemplo (1.10, 1.10).
    """
    if pontos.size == 0:
        return 0.0

    # Mantém apenas pontos que estão dentro da referência.
    pontos = pontos[(pontos[:, 0] < ref[0]) & (pontos[:, 1] < ref[1])]
    if pontos.size == 0:
        return 0.0

    # Remove dominados novamente por segurança.
    nd = []
    for i in range(len(pontos)):
        dominado = False
        for j in range(len(pontos)):
            if i != j and dominates(pontos[j], pontos[i]):
                dominado = True
                break
        if not dominado:
            nd.append(pontos[i])

    pontos = np.array(nd)
    pontos = pontos[np.argsort(pontos[:, 0])]

    hv = 0.0
    y_atual = ref[1]

    for x, y in pontos:
        largura = ref[0] - x
        altura = y_atual - y
        if largura > 0 and altura > 0:
            hv += largura * altura
            y_atual = y

    return float(hv)


def score_fronteira(
    front: List[Individual],
    rmse_ref: float,
    max_terms: int,
    tempo_s: float,
) -> Dict:
    pontos = filtrar_pontos_unicos(front)
    if len(pontos) == 0:
        return {"score": -1e9, "hypervolume": 0.0, "best_rmse": 1e9, "n_nd": 0}

    pontos_norm = np.column_stack([
        pontos[:, 0] / rmse_ref,
        pontos[:, 1] / max_terms,
    ])

    hv = hypervolume_2d_min(pontos_norm, ref=(1.20, 1.20))
    best_rmse = float(np.min(pontos[:, 0]))
    n_nd = len(pontos)

    # Score principal: maximizar hypervolume.
    # Pequena penalização de tempo para favorecer configurações mais baratas em empate.
    score = hv - 0.0001 * tempo_s

    return {
        "score": float(score),
        "hypervolume": float(hv),
        "best_rmse": best_rmse,
        "n_nd": int(n_nd),
    }


def amostrar_parametros_nsga2(n_features: int, tentativa: int, seed_global: int) -> NSGAParams:
    rng = np.random.default_rng(seed_global + tentativa * 101)

    pop_size = int(rng.choice([40, 60, 80, 100, 120]))
    n_gen = int(rng.choice([30, 50, 70, 90]))
    crossover_prob = float(rng.uniform(0.75, 0.98))

    # Mutação recomendada para binário costuma girar em torno de 1/n_features,
    # mas a busca aleatória testa valores um pouco maiores também.
    candidatos_mut = [
        1.0 / n_features,
        2.0 / n_features,
        5.0 / n_features,
        0.01,
        0.02,
        0.05,
    ]
    mutation_prob_gene = float(rng.choice(candidatos_mut))
    mutation_prob_gene = min(max(mutation_prob_gene, 1.0 / n_features), 0.20)

    init_prob_gene = float(rng.uniform(0.05, 0.30))
    tournament_size = int(rng.choice([2, 3]))
    seed = int(seed_global + 1000 + tentativa)

    return NSGAParams(
        pop_size=pop_size,
        n_gen=n_gen,
        crossover_prob=crossover_prob,
        mutation_prob_gene=mutation_prob_gene,
        init_prob_gene=init_prob_gene,
        tournament_size=tournament_size,
        seed=seed,
    )


def busca_aleatoria_nsga2(
    evaluator: Evaluator,
    n_features: int,
    n_tentativas: int,
    seed_global: int,
    max_terms: int,
) -> Tuple[NSGAParams, pd.DataFrame]:
    resultados = []
    melhor_params = None
    melhor_score = -1e18

    print("\n============================================================")
    print("BUSCA ALEATÓRIA DOS HIPERPARÂMETROS DO NSGA-II")
    print("============================================================")

    for t in range(n_tentativas):
        params = amostrar_parametros_nsga2(n_features, t, seed_global)

        print(
            f"\nTentativa {t + 1}/{n_tentativas} | "
            f"pop={params.pop_size}, gen={params.n_gen}, "
            f"pc={params.crossover_prob:.3f}, pm={params.mutation_prob_gene:.5f}, "
            f"pinit={params.init_prob_gene:.3f}, torneio={params.tournament_size}"
        )

        pop, info = executar_nsga2(evaluator, n_features, params, verbose=False)
        front = obter_frente_nao_dominada(pop)
        metricas = score_fronteira(front, evaluator.rmse_ref, max_terms, info["tempo_s"])

        linha = {
            "tentativa": t + 1,
            "score": metricas["score"],
            "hypervolume": metricas["hypervolume"],
            "best_rmse": metricas["best_rmse"],
            "n_nd": metricas["n_nd"],
            "tempo_s": info["tempo_s"],
            "pop_size": params.pop_size,
            "n_gen": params.n_gen,
            "crossover_prob": params.crossover_prob,
            "mutation_prob_gene": params.mutation_prob_gene,
            "init_prob_gene": params.init_prob_gene,
            "tournament_size": params.tournament_size,
            "seed": params.seed,
        }
        resultados.append(linha)

        print(
            f"Resultado: score={metricas['score']:.5f}, "
            f"HV={metricas['hypervolume']:.5f}, "
            f"melhor RMSE={metricas['best_rmse']:.6f}, "
            f"não dominados={metricas['n_nd']}, "
            f"tempo={info['tempo_s']:.1f}s"
        )

        if metricas["score"] > melhor_score:
            melhor_score = metricas["score"]
            melhor_params = params

    df_resultados = pd.DataFrame(resultados).sort_values("score", ascending=False)

    print("\n============================================================")
    print("MELHOR CONFIGURAÇÃO ENCONTRADA")
    print("============================================================")
    print(df_resultados.head(1).T)

    return melhor_params, df_resultados


# ============================================================
# 10. SELEÇÃO DO JOELHO DA FRONTEIRA
# ============================================================

def selecionar_joelho(front: List[Individual]) -> Individual:
    """
    Seleciona uma solução de compromisso pela menor distância ao ponto ideal normalizado.
    Ideal: menor RMSE e menor número de termos.
    """
    pontos = np.array([ind.objectives for ind in front], dtype=float)

    mins = pontos.min(axis=0)
    maxs = pontos.max(axis=0)
    denom = maxs - mins
    denom[denom == 0] = 1.0

    norm = (pontos - mins) / denom
    dist = np.sqrt(np.sum(norm ** 2, axis=1))
    idx = int(np.argmin(dist))
    return front[idx]


def estimar_coeficientes_finais(
    evaluator: Evaluator,
    mask: np.ndarray,
    feature_names: List[str],
) -> pd.DataFrame:
    """
    Reestima os coeficientes do modelo selecionado usando os dados normalizados de treino.
    Retorna coeficientes associados às features selecionadas.
    """
    mask = evaluator.reparar_mask(mask)
    idx = np.where(mask)[0]

    Xtr_sel = evaluator.X_train[:, idx]
    ytr = evaluator.y_train

    Xtr = np.column_stack([np.ones(Xtr_sel.shape[0]), Xtr_sel])
    n_coef = Xtr.shape[1]
    I = np.eye(n_coef)
    I[0, 0] = 0.0

    A = Xtr.T @ Xtr + evaluator.ridge_alpha * I
    b = Xtr.T @ ytr

    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b

    nomes = ["intercepto"] + [feature_names[i] for i in idx]

    return pd.DataFrame({
        "termo": nomes,
        "coeficiente_modelo_normalizado": beta,
    })


# ============================================================
# 11. PLOTS E EXPORTAÇÃO
# ============================================================

def plotar_fronteira(front: List[Individual], ind_joelho: Individual, pasta_saida: str) -> None:
    pontos = np.array([ind.objectives for ind in front], dtype=float)
    pontos = pontos[np.argsort(pontos[:, 1])]

    plt.figure(figsize=(9, 6))
    plt.scatter(pontos[:, 1], pontos[:, 0], label="Soluções não dominadas")
    plt.scatter(
        [ind_joelho.objectives[1]],
        [ind_joelho.objectives[0]],
        s=120,
        marker="*",
        label="Solução de compromisso"
    )
    plt.xlabel("Complexidade do modelo: número de termos")
    plt.ylabel("RMSE de validação")
    plt.title("Fronteira de Pareto - NSGA-II para seleção NARX")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    caminho = os.path.join(pasta_saida, "fronteira_pareto_rmse_complexidade.png")
    plt.savefig(caminho, dpi=300)
    plt.show()

    print(f"Figura salva em: {caminho}")


def exportar_resultados(
    front: List[Individual],
    ind_joelho: Individual,
    evaluator: Evaluator,
    feature_names: List[str],
    df_busca: pd.DataFrame,
    historico_final: pd.DataFrame,
    pasta_saida: str,
) -> None:
    garantir_pasta(pasta_saida)

    linhas = []
    for i, ind in enumerate(front):
        detalhes = evaluator.detalhes(ind.mask)
        linhas.append({
            "id": i,
            "rmse_val": float(ind.objectives[0]),
            "n_terms": int(ind.objectives[1]),
            "rmse_train": detalhes.get("rmse_train", np.nan),
            "mae_val": detalhes.get("mae_val", np.nan),
            "gap_train_val": detalhes.get("gap_train_val", np.nan),
            "features": " | ".join(detalhes.get("selected_features", [])),
            "eh_joelho": bool(np.array_equal(ind.mask, ind_joelho.mask)),
        })

    df_front = pd.DataFrame(linhas).sort_values(["n_terms", "rmse_val"])
    df_coef = estimar_coeficientes_finais(evaluator, ind_joelho.mask, feature_names)

    caminho_excel = os.path.join(pasta_saida, "resultados_nsga2_narx.xlsx")

    with pd.ExcelWriter(caminho_excel) as writer:
        df_busca.to_excel(writer, sheet_name="busca_aleatoria", index=False)
        df_front.to_excel(writer, sheet_name="fronteira_pareto", index=False)
        df_coef.to_excel(writer, sheet_name="modelo_joelho_coef", index=False)
        historico_final.to_excel(writer, sheet_name="historico_final", index=False)

    print(f"Resultados salvos em: {caminho_excel}")


# ============================================================
# 12. PIPELINE PRINCIPAL
# ============================================================

def carregar_dados(config: Dict) -> pd.DataFrame:
    arquivo = config["ARQUIVO_DADOS"]

    if os.path.exists(arquivo):
        print(f"Arquivo encontrado: {arquivo}")
        if arquivo.lower().endswith(".csv"):
            df = pd.read_csv(arquivo)
        else:
            df = pd.read_excel(arquivo)
    else:
        if not config["USAR_DADOS_EXEMPLO_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")
        print(f"Arquivo {arquivo} não encontrado. Gerando dados de exemplo...")
        df = gerar_dados_exemplo(seed=config["SEED_GLOBAL"])

    return df


def preparar_dados(config: Dict):
    df = carregar_dados(config)

    coluna_saida = config["COLUNA_SAIDA"]
    colunas_entrada = config["COLUNAS_ENTRADA"]

    resolver_colunas(df, coluna_saida, colunas_entrada)

    # Remove linhas com NaN nas colunas usadas.
    df = df[[*colunas_entrada, coluna_saida]].dropna().reset_index(drop=True)

    X_base, y = criar_matriz_lags(
        df,
        coluna_saida=coluna_saida,
        colunas_entrada=colunas_entrada,
        max_lag_y=config["MAX_LAG_Y"],
        max_lag_u=config["MAX_LAG_U"],
    )

    X_poly = criar_features_polinomiais(
        X_base,
        grau=config["GRAU_POLINOMIAL"],
        max_features=config["MAX_FEATURES_POLINOMIAIS"],
        seed=config["SEED_GLOBAL"],
    )

    X = X_poly.values.astype(float)
    y_arr = y.values.astype(float)
    feature_names = list(X_poly.columns)

    n = len(y_arr)
    n_train = int(config["FRAC_TREINO"] * n)

    X_train = X[:n_train]
    y_train = y_arr[:n_train]
    X_val = X[n_train:]
    y_val = y_arr[n_train:]

    X_train_norm, X_val_norm, mu, sigma = normalizar_treino_validacao(X_train, X_val)

    print("\n============================================================")
    print("DADOS PREPARADOS")
    print("============================================================")
    print(f"Amostras totais após lags: {n}")
    print(f"Treino: {len(y_train)} | Validação: {len(y_val)}")
    print(f"Número de features candidatas: {len(feature_names)}")
    print(f"Máximo de termos por modelo: {config['MAX_TERMOS_MODELO']}")

    return X_train_norm, y_train, X_val_norm, y_val, feature_names, mu, sigma


def executar_pipeline(config: Dict) -> Dict:
    set_seed(config["SEED_GLOBAL"])
    garantir_pasta(config["PASTA_SAIDA"])

    X_train, y_train, X_val, y_val, feature_names, mu, sigma = preparar_dados(config)
    n_features = len(feature_names)

    evaluator = Evaluator(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        feature_names=feature_names,
        ridge_alpha=config["RIDGE_ALPHA"],
        max_terms=config["MAX_TERMOS_MODELO"],
    )

    print(f"RMSE de referência pela média do treino: {evaluator.rmse_ref:.6f}")

    melhor_params, df_busca = busca_aleatoria_nsga2(
        evaluator=evaluator,
        n_features=n_features,
        n_tentativas=config["N_TENTATIVAS_BUSCA_ALEATORIA"],
        seed_global=config["SEED_GLOBAL"],
        max_terms=config["MAX_TERMOS_MODELO"],
    )

    # Roda final com melhor configuração, aumentando o número de gerações.
    params_final = NSGAParams(
        pop_size=melhor_params.pop_size,
        n_gen=int(melhor_params.n_gen * config["MULTIPLICADOR_GERACOES_FINAL"]),
        crossover_prob=melhor_params.crossover_prob,
        mutation_prob_gene=melhor_params.mutation_prob_gene,
        init_prob_gene=melhor_params.init_prob_gene,
        tournament_size=melhor_params.tournament_size,
        seed=melhor_params.seed + 777,
    )

    print("\n============================================================")
    print("EXECUÇÃO FINAL DO NSGA-II COM MELHORES HIPERPARÂMETROS")
    print("============================================================")
    print(params_final)

    pop_final, info_final = executar_nsga2(evaluator, n_features, params_final, verbose=True)
    front_final = obter_frente_nao_dominada(pop_final)
    ind_joelho = selecionar_joelho(front_final)
    detalhes_joelho = evaluator.detalhes(ind_joelho.mask)

    print("\n============================================================")
    print("MODELO DE COMPROMISSO SELECIONADO")
    print("============================================================")
    print(f"RMSE validação: {ind_joelho.objectives[0]:.6f}")
    print(f"MAE validação : {detalhes_joelho.get('mae_val', np.nan):.6f}")
    print(f"RMSE treino   : {detalhes_joelho.get('rmse_train', np.nan):.6f}")
    print(f"Gap treino-val: {detalhes_joelho.get('gap_train_val', np.nan):.6f}")
    print(f"Nº de termos  : {int(ind_joelho.objectives[1])}")
    print("\nTermos selecionados:")
    for termo in detalhes_joelho.get("selected_features", []):
        print(f"  - {termo}")

    plotar_fronteira(front_final, ind_joelho, config["PASTA_SAIDA"])

    exportar_resultados(
        front=front_final,
        ind_joelho=ind_joelho,
        evaluator=evaluator,
        feature_names=feature_names,
        df_busca=df_busca,
        historico_final=info_final["historico"],
        pasta_saida=config["PASTA_SAIDA"],
    )

    return {
        "evaluator": evaluator,
        "feature_names": feature_names,
        "melhor_params_busca": melhor_params,
        "params_final": params_final,
        "pop_final": pop_final,
        "front_final": front_final,
        "ind_joelho": ind_joelho,
        "df_busca": df_busca,
        "historico_final": info_final["historico"],
        "mu_features": mu,
        "sigma_features": sigma,
    }


# ============================================================
# 13. EXECUTAR
# ============================================================

if __name__ == "__main__":
    resultados = executar_pipeline(CONFIG)
